# Comparing Glider Observations with Satellite Data
### Authors: Madison Richardson, Cara Wilson, & Dale Robinson

> History | Updated Sept 2026. Converted from R Markdown to a Jupyter notebook.

This tutorial demonstrates how to compare in situ ocean glider observations with
satellite-derived oceanographic measurements using R.

Glider observations from CalCOFI Line 90 are retrieved from the Scripps Spray
Glider ERDDAP server. Corresponding satellite observations are then retrieved
from NOAA CoastWatch ERDDAP servers and matched to the glider observations in
space and time.

The comparison includes:

- Chlorophyll-a
- Sea surface temperature (SST)
- Sea surface salinity (SSS)

---

**File:** `notebooks/04_glider_satellite_r.ipynb`

**What this does:** Retrieves a glider track and the satellite measurements
along it, then plots the two against each other.

**How to run it:** Open in JupyterLab, check the kernel in the top-right corner
says **R**, then choose *Run > Run All Cells*. The satellite matching steps take
a few minutes.

**Inputs:** none -- everything is read from public ERDDAP servers.

**Outputs:** three comparison plots, one per variable.

## Load packages

The first block installs anything missing, then loads all four packages. If you
have already run `Rscript install.R`, nothing needs installing and this is quick.

In [ ]:
packages <- c( "rerddap", "rerddapXtracto","tidyverse","scales")


# Install packages not yet installed
installed_packages <- packages %in% rownames(installed.packages())

if (any(installed_packages == FALSE)) {
  install.packages(packages[!installed_packages])
}

# Load packages 
invisible(lapply(packages, library, character.only = TRUE))

Jupyter needs to be told how large to draw R plots. This cell is the only
addition to the original tutorial -- it affects display size only.

In [ ]:
# Display size for the plots below (Jupyter-specific).
options(repr.plot.width = 10, repr.plot.height = 5)

## Retrieve Glider Observations

Glider data along **CalCOFI Line 90**, pulled from the Scripps Spray Glider
ERDDAP server with `tabledap`, covering January to June 2026.

In [ ]:
#Setup erddap dataset info 

url <- "https://spraydata.ucsd.edu/erddap/"
dataset_info <- info("binnedCUGN90", url = url)

titletext <- "CalCOFI Line 90" #change if glider dataset name is changed, used on the plots produced  

# Fetch the data using tabledap
glider <- tabledap(
  dataset_info,
  fields = c("longitude", "latitude", "time", "depth", "chlorophyll", "temperature", "salinity", "doxy"),
  "time>=2026-01-01",
  "time<=2026-06-30"
)

### Select Surface Observations

A satellite only sees the surface, so keep the glider readings at 10 m depth.
Those are the rows we can fairly compare against satellite pixels.

In [ ]:
glider_surface <- subset(glider, depth==10) 

## Retrieve Satellite Chlorophyll

`rxtracto()` does the real work: for every point on the glider track it goes to
the satellite dataset and pulls the values in a small box (0.2 x 0.2 degrees)
around that position, at that date. That is the space-and-time matching.

In [ ]:
# Get data information from ERDDAP server 
dataset <- 'pmlEsaCCI60OceanColorDaily'
dataInfo <- rerddap::info(dataset, url= "https://coastwatch.pfeg.noaa.gov/erddap")

# Set the variable we want to extract data from:
parameter <- 'chlor_a'

# Set xlen, ylen to 0.2 degree
xlen <- 0.2 
ylen <- 0.2


# Get variables x, y, t coordinates from glider data
xcoords <- glider_surface$longitude
ycoords <- glider_surface$latitude
tcoords <- as.Date(glider_surface$time)

# Extract satellite data 
chl_sat <- rxtracto(dataInfo, 
                  parameter=parameter, 
                  xcoord=xcoords, ycoord=ycoords, 
                  tcoord=tcoords, xlen=xlen, ylen=ylen)

## Retrieve Satellite Sea Surface Temperature

Same matching, different dataset.

In [ ]:
# Get data information from ERDDAP server 
dataset <- 'noaacwLEOACSPOSSTL3SCWeeklyNRT'
dataInfo <- rerddap::info(dataset, url= "https://coastwatch.pfeg.noaa.gov/erddap")

# Set the variable we want to extract data from:
parameter <- 'sea_surface_temperature'

# Extract satellite data 
sst_sat <- rxtracto(dataInfo, 
                  parameter=parameter, 
                  xcoord=xcoords, ycoord=ycoords, 
                  tcoord=tcoords, xlen=xlen, ylen=ylen)

## Retrieve Satellite Sea Surface Salinity

Same again, with one wrinkle: this dataset has an altitude dimension, so
`rxtracto()` is given a `zcoord` as well.

In [ ]:
# Get data information from ERDDAP server 
dataset <- 'coastwatchSMOSv662SSS3day'
dataInfo <- rerddap::info(dataset, url= "https://coastwatch.pfeg.noaa.gov/erddap")

# Set the variable we want to extract data from:
parameter <- 'sss'

# Extract satellite data. This dataset has an altitude dimension so we have to specify that in the rxtracto call by giving it zcoord   
salinity_sat <- rxtracto(dataInfo, 
                  parameter=parameter, 
                  xcoord=xcoords, ycoord=ycoords, zcoord=ycoords*0,
                  tcoord=tcoords, xlen=xlen, ylen=ylen)

## Combine Glider and Satellite Observations

Each `rxtracto()` result comes back in the same order as the track, so the
matched satellite values can be attached straight onto the glider data frame.

In [ ]:
glider_surface$chl_sat <- chl_sat$'mean chlor_a'
glider_surface$sst_sat <- sst_sat$'mean sea_surface_temperature'
glider_surface$salinity_sat <- salinity_sat$'mean sss'

## Compare the Two

Each plot below reshapes the data so glider and satellite become two series of
one variable, then draws them together. Where the lines track each other, the
satellite is seeing what the glider measured.

### Chlorophyll-a

In [ ]:
# 1. Reshape data for a unified legend
glider_surface %>%
  pivot_longer(
    cols = c(chlorophyll, chl_sat),
    names_to = "source",
    values_to = "sal_val"
  ) %>%
  mutate(source = ifelse(source == "chlorophyll", "Glider (In Situ)", "Satellite (Surface)")) %>%

# 2. Build the upgraded plot
  ggplot(aes(x = time, y = sal_val, color = source)) +
  # Background trend lines to connect time points
  geom_line(alpha = 0.35, linewidth = 0.5) +
  # Points with transparency to handle overlapping observations
  geom_point(alpha = 0.7, size = 1.5, na.rm = TRUE) +
  scale_color_manual(values = c("Glider (In Situ)" = "forestgreen", "Satellite (Surface)" = "darkgray")) +
  # Format time axis neatly
  scale_x_datetime(
    breaks = breaks_pretty(n = 10),      # Automatically targets ~10 neatly spaced ticks
    labels = label_date_short()          # Concise format (e.g., "Jan", "15", "Feb") without repeating years
  ) +
  # Dynamically include your title_text variable inside labs()
  labs(
    title = paste("Chlorophyll Comparison:", titletext),
    subtitle = "Glider surface measurements vs. satellite observations",
    x = NULL,
    y = "Chlorophyll",
    color = "Data Source"
  ) +
  # Minimalist oceanographic styling
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    plot.subtitle = element_text(color = "grey40", margin = margin(b = 10)),
    legend.position = "top",
    legend.title = element_text(face = "bold"),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_line(color = "grey90", linetype = "dashed"),
    axis.text.x = element_text(angle = 30, hjust = 1)
  )

### Sea Surface Temperature

In [ ]:
# 1. Reshape data for a unified legend & clean mapping
glider_surface %>%
  pivot_longer(
    cols = c(temperature, sst_sat),
    names_to = "source",
    values_to = "temp_val"
  ) %>%
  mutate(source = ifelse(source == "temperature", "Glider (In Situ)", "Satellite (SST)")) %>%
  
# 2. Build the upgraded plot
  ggplot(aes(x = time, y = temp_val, color = source)) +
  # Background trend lines to help connect time points
  geom_line(alpha = 0.35, linewidth = 0.5) +
  # Points with custom size and transparency
  geom_point(alpha = 0.7, size = 1.5, na.rm = TRUE) +
  scale_color_manual(values = c("Glider (In Situ)" = "blue", "Satellite (SST)" = "darkgray")) +
  # Format date axis neatly (e.g., "Jan 2025")
  scale_x_datetime(
    breaks = breaks_pretty(n = 10),      # Automatically targets ~10 neatly spaced ticks
    labels = label_date_short()          # Concise format (e.g., "Jan", "15", "Feb") without repeating years
  ) +
  # Labels and units
  labs(
    title = paste("Sea Surface Temperature Comparison for",titletext),
    subtitle = "Glider surface measurements vs. satellite observations",
    x = NULL,
    y = "Temperature (°C)",
    color = "Data Source"
  ) +
  # Minimalist oceanographic styling
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    plot.subtitle = element_text(color = "grey40", margin = margin(b = 10)),
    legend.position = "top",
    legend.title = element_text(face = "bold"),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_line(color = "grey90", linetype = "dashed"),
    axis.text.x = element_text(angle = 30, hjust = 1)
  )

### Sea Surface Salinity

In [ ]:
# 1. Reshape data for a unified legend
glider_surface %>%
  pivot_longer(
    cols = c(salinity, salinity_sat),
    names_to = "source",
    values_to = "sal_val"
  ) %>%
  mutate(source = ifelse(source == "salinity", "Glider (In Situ)", "Satellite (SSS)")) %>%

# 2. Build the upgraded plot
  ggplot(aes(x = time, y = sal_val, color = source)) +
  # Background trend lines to connect time points
  geom_line(alpha = 0.35, linewidth = 0.5) +
  # Points with transparency to handle overlapping observations
  geom_point(alpha = 0.7, size = 1.5, na.rm = TRUE) +
  # High-contrast oceanographic palette (Navy/Blue for Glider, Amber/Orange for Satellite)
  scale_color_manual(values = c("Glider (In Situ)" = "#E76F51", "Satellite (SSS)" = "darkgray")) +
  # Format time axis neatly
  scale_x_datetime(
    breaks = breaks_pretty(n = 10),      # Automatically targets ~10 neatly spaced ticks
    labels = label_date_short()          # Concise format (e.g., "Jan", "15", "Feb") without repeating years
  ) +
  # Dynamically include your title_text variable inside labs()
  labs(
    title = paste("Sea Surface Salinity Comparison:", titletext),
    subtitle = "Glider surface measurements vs. satellite observations",
    x = NULL,
    y = "Salinity (PSU)",
    color = "Data Source"
  ) +
  # Minimalist oceanographic styling
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    plot.subtitle = element_text(color = "grey40", margin = margin(b = 10)),
    legend.position = "top",
    legend.title = element_text(face = "bold"),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_line(color = "grey90", linetype = "dashed"),
    axis.text.x = element_text(angle = 30, hjust = 1)
  )

## Summary

You retrieved a glider track, pulled the matching satellite observations along
it, and compared the two for three variables.

The Python version of this tutorial, `04_glider_satellite_python.ipynb`, does
the same thing with `erddapy` and `xarray` -- worth a look if you want to see
the same problem solved in the other language.

Next: **`05_hackathon_data_r.ipynb`**, which opens the hackathon's own
data files.